In [10]:
import zipfile
import shutil
from pathlib import Path
import csv
import json
import hashlib

import numpy as np
from PIL import Image, UnidentifiedImageError
import pytesseract
import cv2
import xml.etree.ElementTree as ET


# =====================================================================
# CONFIG
# =====================================================================

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

OCR_CONF_THRESHOLD = 60
MIN_CONTRAST_PASS = 4.5      # baseline WCAG AA for normal text
MIN_TEXT_HEIGHT_PX = 14      # smallish; we will treat >= LARGE_TEXT_PX as "large text"
BLUR_THRESHOLD = 80.0

LARGE_TEXT_PX = 18           # heuristic: if OCR box height >= 18 px, treat as "large text"
DPI_MIN_PRINT = 300          # typical print-quality DPI threshold


# =====================================================================
# WCAG FUNCTIONS
# =====================================================================

def srgb_to_linear(c):
    if c <= 0.04045:
        return c / 12.92
    return ((c + 0.055) / 1.055) ** 2.4


def relative_luminance(rgb):
    r, g, b = [x / 255.0 for x in rgb]
    return (
        0.2126 * srgb_to_linear(r) +
        0.7152 * srgb_to_linear(g) +
        0.0722 * srgb_to_linear(b)
    )


def contrast_ratio(c1, c2):
    L1 = relative_luminance(c1)
    L2 = relative_luminance(c2)
    return (max(L1, L2) + 0.05) / (min(L1, L2) + 0.05)


# =====================================================================
# WORD IMAGE EXTRACTION
# =====================================================================

def extract_word_images(docx_path, out_dir):
    docx_path = Path(docx_path)
    out_dir = Path(out_dir)
    media_dir = out_dir / "word_media"
    media_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with zipfile.ZipFile(docx_path, "r") as z:
        for name in z.namelist():
            if name.startswith("word/media/"):
                filename = Path(name).name
                target = media_dir / filename
                with z.open(name) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                extracted.append(target)

    print(f"[INFO] Extracted {len(extracted)} Word images → {media_dir}")
    return extracted, media_dir


def extract_word_alt_text(docx_path):
    docx_path = Path(docx_path)
    alt_map = {}

    with zipfile.ZipFile(docx_path, "r") as z:
        ns = {
            "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
            "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
            "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
            "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture",
        }

        # rId → image file
        rels = {}
        rel_xml = z.read("word/_rels/document.xml.rels")
        rel_root = ET.fromstring(rel_xml)
        rel_ns = {"": "http://schemas.openxmlformats.org/package/2006/relationships"}

        for r in rel_root.findall("Relationship", rel_ns):
            if r.attrib.get("Type", "").endswith("/image"):
                rels[r.attrib["Id"]] = Path(r.attrib["Target"]).name

        doc_xml = z.read("word/document.xml")
        root = ET.fromstring(doc_xml)

        for pic in root.findall(".//pic:pic", ns):
            cnvpr = pic.find("pic:nvPicPr/pic:cNvPr", ns)
            if cnvpr is None:
                continue
            alt_text = cnvpr.attrib.get("descr", "")
            blip = pic.find(".//a:blip", ns)
            if blip is not None:
                embed = blip.attrib.get(
                    "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}embed"
                )
                if embed in rels:
                    img_name = rels[embed]
                    alt_map[img_name] = alt_text

    print(f"[INFO] Alt text entries for {len(alt_map)} images")
    return alt_map


# =====================================================================
# DUPLICATE DETECTION
# =====================================================================

def image_fingerprint(path, size=(32, 32)):
    try:
        img = Image.open(path).convert("L")
    except Exception:
        return None  # unreadable
    img = img.resize(size)
    arr = np.array(img, dtype=np.uint8)
    return hashlib.sha256(arr.tobytes()).hexdigest()


def find_duplicates(paths):
    fp_map = {}
    groups = {}

    for p in paths:
        fp = image_fingerprint(p)
        fp_map[p] = fp
        groups.setdefault(fp, []).append(p)

    return fp_map, groups


# =====================================================================
# COLORBLIND / GRAYSCALE RISK (heuristic)
# =====================================================================

def compute_color_risks(arr):
    """
    Heuristic:
      - Quantize colors into coarse bins.
      - If there are many distinct colors but they collapse into very few
        grayscale buckets, we flag Risk.
    """
    h, w, _ = arr.shape

    # Downsample for speed
    scale = max(h / 200.0, w / 200.0, 1.0)
    if scale > 1.0:
        new_w = int(w / scale)
        new_h = int(h / scale)
        arr_small = cv2.resize(arr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    else:
        arr_small = arr

    colors = arr_small.reshape(-1, 3)
    if colors.size == 0:
        return "OK", "OK", "No pixels", "No pixels"

    # Quantize RGB to 8 levels per channel
    quant = (colors // 32).astype(np.uint8)  # 0..7
    uniques = np.unique(quant, axis=0)
    n_colors = len(uniques)

    # Luminance-grayscale for each quantized color
    r = uniques[:, 0].astype(float)
    g = uniques[:, 1].astype(float)
    b = uniques[:, 2].astype(float)
    grays = 0.2126 * r + 0.7152 * g + 0.0722 * b

    gray_bins = (grays // 32).astype(int)  # 0..7
    n_gray = len(np.unique(gray_bins))

    if n_colors >= 4 and n_gray <= 2:
        cb_risk = "Risk"
        gs_risk = "Risk"
        cb_reason = (
            f"Risk: {n_colors} color bins collapse to {n_gray} grayscale bins; "
            "different hues may look similar to colorblind users."
        )
        gs_reason = (
            f"Risk: {n_colors} color bins collapse to {n_gray} grayscale bins; "
            "patterns may not be distinguishable in grayscale."
        )
    else:
        cb_risk = "OK"
        gs_risk = "OK"
        cb_reason = (
            f"OK: {n_colors} color bins map to {n_gray} grayscale bins; "
            "likely retains distinctions under colorblind vision."
        )
        gs_reason = (
            f"OK: {n_colors} color bins map to {n_gray} grayscale bins; "
            "likely retains distinctions when printed in grayscale."
        )

    return cb_risk, gs_risk, cb_reason, gs_reason


# =====================================================================
# ADA ANALYSIS
# =====================================================================

def analyze_image_for_ada(image_path):
    """
    Analyze a single image for ADA/WCAG compliance using regional contrast sampling.
    For each OCR word box, we:
      - take the RGB patch under the box
      - compute grayscale
      - split pixels into 3 bands (dark / mid / light) via percentiles
      - treat the darkest band as text (foreground)
      - other bands as background(s)
      - compute contrast between foreground and each background band
      - use the minimum of these contrasts for that word

    The image-level min_contrast is the minimum contrast over all words.
    """
    img_path = Path(image_path)

    try:
        img = Image.open(img_path)
    except Exception as e:
        raise e

    # ----- DPI -----
    dpi_val = None
    if "dpi" in img.info:
        dpi_info = img.info["dpi"]
        if isinstance(dpi_info, tuple) and len(dpi_info) > 0:
            dpi_val = float(dpi_info[0])
        elif isinstance(dpi_info, (int, float)):
            dpi_val = float(dpi_info)

    img = img.convert("RGB")
    arr = np.array(img)
    h, w, _ = arr.shape

    # ----- OCR -----
    ocr = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

    has_text = False
    min_contrast = None
    min_text_height = None

    def local_contrast_for_box(x, y, bw, bh):
        """
        Compute local worst-case contrast in this bounding box using
        3-band grayscale segmentation.
        Returns contrast (float) or None if we can't segment.
        """
        # Clip to image bounds
        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(w, x + bw)
        y2 = min(h, y + bh)
        if x2 <= x1 or y2 <= y1:
            return None

        patch = arr[y1:y2, x1:x2, :]  # H x W x 3
        patch_gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY).astype(np.float32)
        flat_gray = patch_gray.reshape(-1)

        # Not enough pixels or too uniform -> can't reliably segment
        if flat_gray.size < 50 or np.std(flat_gray) < 5.0:
            return None

        # 3 approximate bands: dark / mid / light
        p33 = np.percentile(flat_gray, 33)
        p66 = np.percentile(flat_gray, 66)

        band0_mask = flat_gray <= p33          # darkest
        band1_mask = (flat_gray > p33) & (flat_gray <= p66)
        band2_mask = flat_gray > p66          # lightest

        bands = []
        for mask in (band0_mask, band1_mask, band2_mask):
            idx = np.where(mask)[0]
            if idx.size == 0:
                bands.append(None)
                continue
            rgb_vals = patch.reshape(-1, 3)[idx]
            mean_rgb = rgb_vals.mean(axis=0)
            mean_gray = patch_gray.reshape(-1)[idx].mean()
            bands.append((mean_rgb, mean_gray))

        # Require at least 2 non-empty bands
        valid_bands = [b for b in bands if b is not None]
        if len(valid_bands) < 2:
            return None

        # Sort by gray ascending: darkest is text, others backgrounds
        valid_bands.sort(key=lambda b: b[1])
        fg_rgb, fg_gray = valid_bands[0]
        bg_bands = valid_bands[1:]

        # Compute contrast between text band and each background band;
        # take the minimum (worst-case).
        local_min = None
        for bg_rgb, bg_gray in bg_bands:
            cr = contrast_ratio(fg_rgb, bg_rgb)
            if local_min is None or cr < local_min:
                local_min = cr

        return local_min

    # ----- Loop over OCR words -----
    n = len(ocr["text"])
    for i in range(n):
        text = ocr["text"][i].strip()
        try:
            conf = int(ocr["conf"][i])
        except ValueError:
            continue
        if not text or conf < OCR_CONF_THRESHOLD:
            continue

        x = ocr["left"][i]
        y = ocr["top"][i]
        bw = ocr["width"][i]
        bh = ocr["height"][i]

        # local contrast using regional sampling
        local_cr = local_contrast_for_box(x, y, bw, bh)
        if local_cr is None:
            # couldn't get a reliable local contrast for this box; skip
            continue

        has_text = True

        # track smallest text height
        if min_text_height is None or bh < min_text_height:
            min_text_height = bh

        # track minimum contrast across all words
        if min_contrast is None or local_cr < min_contrast:
            min_contrast = local_cr

    # ----- Blur detection -----
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blur_metric = cv2.Laplacian(gray, cv2.CV_64F).var()

    # ----- DPI check -----
    if dpi_val is None:
        dpi_pass = None
        dpi_reason = "No embedded DPI; manual print-quality check recommended."
    else:
        if dpi_val >= DPI_MIN_PRINT:
            dpi_pass = True
            dpi_reason = f"Pass: {dpi_val:.0f} ≥ {DPI_MIN_PRINT} dpi."
        else:
            dpi_pass = False
            dpi_reason = f"Fail: {dpi_val:.0f} < {DPI_MIN_PRINT} dpi."

    # ----- Colorblind / grayscale risk (unchanged) -----
    cb_risk, gs_risk, cb_reason, gs_reason = compute_color_risks(arr)

    # ----- Build reasoned flags -----
    blur_ok = (blur_metric >= BLUR_THRESHOLD)
    blur_reason = (
        f"Pass: blur metric {blur_metric:.1f} ≥ {BLUR_THRESHOLD:.1f} (sharp enough)."
        if blur_ok
        else f"Fail: blur metric {blur_metric:.1f} < {BLUR_THRESHOLD:.1f} (may be too blurry)."
    )

    # Defaults if no usable text contrast found
    if not has_text or min_contrast is None or min_text_height is None:
        contrast_ok = True
        contrast_reason = "No reliable text contrast detected; contrast check not applicable."
        text_size_ok = True
        text_size_reason = "No reliable text boxes detected; text size check not applicable."
        print_ada_pass = True
        print_ada_reason = "No text detected; print contrast for text not applicable."
        digital_ada_pass = True
        digital_ada_reason = "No text detected; WCAG contrast for text not applicable."
    else:
        is_large_text = (min_text_height >= LARGE_TEXT_PX)

        # Contrast OK against baseline 4.5
        contrast_ok = (min_contrast >= MIN_CONTRAST_PASS)
        if contrast_ok:
            contrast_reason = (
                f"Pass: minimum text contrast {min_contrast:.2f} ≥ {MIN_CONTRAST_PASS:.1f}."
            )
        else:
            contrast_reason = (
                f"Fail: minimum text contrast {min_contrast:.2f} < {MIN_CONTRAST_PASS:.1f}."
            )

        # Text size
        text_size_ok = (min_text_height >= MIN_TEXT_HEIGHT_PX)
        if text_size_ok:
            text_size_reason = (
                f"Pass: smallest detected text height {min_text_height}px ≥ "
                f"{MIN_TEXT_HEIGHT_PX}px threshold."
            )
        else:
            text_size_reason = (
                f"Fail: smallest detected text height {min_text_height}px < "
                f"{MIN_TEXT_HEIGHT_PX}px threshold."
            )

        # WCAG digital
        required_digital = 3.0 if is_large_text else 4.5
        digital_ada_pass = (min_contrast >= required_digital)
        if digital_ada_pass:
            digital_ada_reason = (
                f"Pass: digital contrast {min_contrast:.2f} ≥ {required_digital:.1f} "
                f"(WCAG AA for {'large' if is_large_text else 'normal'} text)."
            )
        else:
            digital_ada_reason = (
                f"Fail: digital contrast {min_contrast:.2f} < {required_digital:.1f} "
                f"(WCAG AA for {'large' if is_large_text else 'normal'} text)."
            )

        # Print (same thresholds, conservative)
        required_print = required_digital
        print_ada_pass = (min_contrast >= required_print)
        if print_ada_pass:
            print_ada_reason = (
                f"Pass: print contrast {min_contrast:.2f} ≥ {required_print:.1f} "
                f"(matching WCAG threshold for {'large' if is_large_text else 'normal'} text)."
            )
        else:
            print_ada_reason = (
                f"Fail: print contrast {min_contrast:.2f} < {required_print:.1f}."
            )

    ada_pass = (
        (digital_ada_pass if has_text else True) and
        (text_size_ok if text_size_ok is not None else True) and
        blur_ok
    )

    return {
        "has_text": has_text,
        "min_contrast": float(min_contrast) if min_contrast is not None else None,
        "min_text_height_px": int(min_text_height) if min_text_height is not None else None,
        "blur_metric": float(blur_metric),
        "dpi": dpi_val,
        "colorblind_risk": cb_risk,
        "colorblind_reason": cb_reason,
        "grayscale_risk": gs_risk,
        "grayscale_reason": gs_reason,
        "contrast_ok": contrast_ok,
        "contrast_reason": contrast_reason,
        "text_size_ok": text_size_ok,
        "text_size_reason": text_size_reason,
        "blur_ok": blur_ok,
        "blur_reason": blur_reason,
        "dpi_pass": dpi_pass,
        "dpi_reason": dpi_reason,
        "print_ada_pass": print_ada_pass,
        "print_ada_reason": print_ada_reason,
        "digital_ada_pass": digital_ada_pass,
        "digital_ada_reason": digital_ada_reason,
        "ada_pass": ada_pass,
    }

# =====================================================================
# REPORTING
# =====================================================================

def make_json_safe(v):
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, dict):
        return {k: make_json_safe(val) for k, val in v.items()}
    if isinstance(v, list):
        return [make_json_safe(x) for x in v]
    return v


def write_reports(out_dir, results):
    out_dir = Path(out_dir)

    fields = [
        "source",
        "image_file",
        "duplicate_group",
        "duplicate_count",
        "is_representative",
        "has_text",
        "min_contrast",
        "min_text_height_px",
        "blur_metric",
        "dpi",
        "print_ada_pass",
        "print_ada_reason",
        "digital_ada_pass",
        "digital_ada_reason",
        "contrast_ok",
        "contrast_reason",
        "text_size_ok",
        "text_size_reason",
        "blur_ok",
        "blur_reason",
        "dpi_pass",
        "dpi_reason",
        "colorblind_risk",
        "colorblind_reason",
        "grayscale_risk",
        "grayscale_reason",
        "ada_pass",
        "alt_text",
        "bad_image",
        "error",
    ]

    csv_path = out_dir / "ada_report.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fields)
        w.writeheader()
        for r in results:
            w.writerow({k: r.get(k) for k in fields})

    json_path = out_dir / "ada_report.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(results), f, indent=2)

    html_path = out_dir / "ada_report.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'><title>ADA Image Report</title></head><body>")
        f.write("<h1>ADA Image Report</h1>")

        f.write("""
        <h2>Check Summary</h2>
        <ul>
          <li><b>Print ADA:</b> uses WCAG-style contrast thresholds; large text may pass at 3:1, normal text at 4.5:1.</li>
          <li><b>Digital ADA:</b> WCAG 2.x AA — 4.5:1 for normal text, 3.0:1 for large text.</li>
          <li><b>Contrast:</b> numeric minimum contrast found between text and background.</li>
          <li><b>Text Size:</b> based on smallest detected OCR text height (pixels).</li>
          <li><b>Blur:</b> Laplacian variance; lower values mean blurrier images.</li>
          <li><b>DPI:</b> print dots-per-inch, when embedded in the file.</li>
          <li><b>Colorblind Risk:</b> flags when many colors collapse into very few grayscale bins.</li>
          <li><b>Grayscale Risk:</b> flags when distinct colors may become indistinguishable in grayscale.</li>
        </ul>
        """)

        f.write("<table border='1' cellpadding='4' cellspacing='0'>")
        f.write(
            "<tr>"
            "<th>Source</th>"
            "<th>Image</th>"
            "<th>Dup Group</th><th>Count</th><th>Rep?</th>"
            "<th>Print ADA</th>"
            "<th>Digital ADA</th>"
            "<th>Contrast</th>"
            "<th>Text Size</th>"
            "<th>Blur</th>"
            "<th>DPI</th>"
            "<th>Colorblind</th>"
            "<th>Grayscale</th>"
            "<th>Alt Text</th>"
            "<th>Bad?</th>"
            "<th>Error</th>"
            "</tr>"
        )

        for r in results:
            img_rel = r.get("image_rel", r["image_file"])
            f.write("<tr>")

            f.write(f"<td>{r['source']}</td>")
            f.write(
                f"<td><img src='{img_rel}' style='max-width:250px; max-height:200px;'><br>"
                f"{r['image_file']}</td>"
            )
            f.write(f"<td>{r.get('duplicate_group','')}</td>")
            f.write(f"<td>{r.get('duplicate_count','')}</td>")
            f.write(f"<td>{r.get('is_representative','')}</td>")

            # Print ADA
            pad = r.get("print_ada_pass")
            pad_color = "green" if pad else "red"
            f.write(
                f"<td style='color:{pad_color}; font-weight:bold;'>"
                f"{'PASS' if pad else 'FAIL'}<br><span style='font-weight:normal;'>{r.get('print_ada_reason','')}</span></td>"
            )

            # Digital ADA
            dad = r.get("digital_ada_pass")
            dad_color = "green" if dad else "red"
            f.write(
                f"<td style='color:{dad_color}; font-weight:bold;'>"
                f"{'PASS' if dad else 'FAIL'}<br><span style='font-weight:normal;'>{r.get('digital_ada_reason','')}</span></td>"
            )

            # Contrast
            f.write(
                f"<td>{r.get('min_contrast')}<br><span style='font-size:smaller;'>{r.get('contrast_reason','')}</span></td>"
            )

            # Text size
            f.write(
                f"<td>{r.get('min_text_height_px')} px<br><span style='font-size:smaller;'>{r.get('text_size_reason','')}</span></td>"
            )

            # Blur
            f.write(
                f"<td>{r.get('blur_metric')}<br><span style='font-size:smaller;'>{r.get('blur_reason','')}</span></td>"
            )

            # DPI
            dpi_val = r.get("dpi")
            f.write(
                f"<td>{dpi_val if dpi_val is not None else 'n/a'}<br>"
                f"<span style='font-size:smaller;'>{r.get('dpi_reason','')}</span></td>"
            )

            # Colorblind / Grayscale
            f.write(
                f"<td>{r.get('colorblind_risk')}<br><span style='font-size:smaller;'>{r.get('colorblind_reason','')}</span></td>"
            )
            f.write(
                f"<td>{r.get('grayscale_risk')}<br><span style='font-size:smaller;'>{r.get('grayscale_reason','')}</span></td>"
            )

            f.write(
                f"<td>{(r.get('alt_text') or '').replace('<','&lt;').replace('>','&gt;')}</td>"
            )
            f.write(f"<td>{r.get('bad_image')}</td>")
            f.write(f"<td>{r.get('error')}</td>")

            f.write("</tr>")

        f.write("</table></body></html>")

    print("Wrote:", csv_path)
    print("Wrote:", json_path)
    print("Wrote:", html_path)


# =====================================================================
# MAIN PIPELINE
# =====================================================================

def run_ada_pipeline(docx_path, output_root, excel_image_dir=None):
    docx_path = Path(docx_path)
    output_root = Path(output_root)
    out_dir = output_root / docx_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    # WORD images
    word_images, word_media_dir = extract_word_images(docx_path, out_dir)
    alt_map = extract_word_alt_text(docx_path)

    # EXCEL images
    excel_images = []
    if excel_image_dir:
        excel_dir = Path(excel_image_dir)
        excel_images = sorted(excel_dir.glob("*.*"))
        print(f"[INFO] Found {len(excel_images)} Excel images in {excel_dir}")

    excel_local = out_dir / "excel_images"
    excel_local.mkdir(exist_ok=True)

    image_records = []
    all_paths = []

    # Word images
    for p in word_images:
        all_paths.append(p)
        image_records.append({
            "source": "word",
            "path": p,
            "image_file": p.name,
            "image_rel": f"word_media/{p.name}",
            "alt_text": alt_map.get(p.name, ""),
        })

    # Excel images
    for p in excel_images:
        dst = excel_local / p.name
        if not dst.exists():
            shutil.copy2(p, dst)
        all_paths.append(dst)
        image_records.append({
            "source": "excel",
            "path": dst,
            "image_file": dst.name,
            "image_rel": f"excel_images/{dst.name}",
            "alt_text": "",
        })

    print(f"[INFO] Total images to analyze: {len(all_paths)}")

    # Duplicates
    fp_map, groups = find_duplicates(all_paths)
    print(f"[INFO] Unique image groups: {len(groups)}")

    # ADA analysis with skip-bad-images
    ada_by_path = {}
    for p in all_paths:
        try:
            ada = analyze_image_for_ada(p)
            ada["bad_image"] = False
            ada["error"] = ""
        except Exception as e:
            print(f"[WARN] Cannot read image: {p} — {e}")
            ada = {
                "has_text": None,
                "min_contrast": None,
                "min_text_height_px": None,
                "blur_metric": None,
                "dpi": None,
                "colorblind_risk": None,
                "colorblind_reason": "",
                "grayscale_risk": None,
                "grayscale_reason": "",
                "contrast_ok": None,
                "contrast_reason": "",
                "text_size_ok": None,
                "text_size_reason": "",
                "blur_ok": None,
                "blur_reason": "",
                "dpi_pass": None,
                "dpi_reason": "",
                "print_ada_pass": False,
                "print_ada_reason": "Image could not be read.",
                "digital_ada_pass": False,
                "digital_ada_reason": "Image could not be read.",
                "ada_pass": False,
                "bad_image": True,
                "error": str(e),
            }
        ada_by_path[p] = ada

    # Build results
    results = []
    for fp, paths in groups.items():
        count = len(paths)
        rep_path = paths[0]

        for p in paths:
            rec = next(r for r in image_records if r["path"] == p)
            ada = ada_by_path[p]

            row = {
                "source": rec["source"],
                "image_file": rec["image_file"],
                "image_rel": rec["image_rel"],
                "duplicate_group": fp if count > 1 else "",
                "duplicate_count": count,
                "is_representative": (p == rep_path),
                "alt_text": rec.get("alt_text", ""),
            }
            row.update(ada)
            results.append(row)

    write_reports(out_dir, results)

    print("\n[COMPLETE] ADA pipeline finished.")
    print("Output folder:", out_dir)


# =====================================================================
# MAIN
# =====================================================================

if __name__ == "__main__":
    DOCX_FILE = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25.docx"
    OUTPUT_ROOT = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\ada_output"
    EXCEL_IMAGE_DIR = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\excel_images"

    run_ada_pipeline(DOCX_FILE, OUTPUT_ROOT, excel_image_dir=EXCEL_IMAGE_DIR)


[INFO] Extracted 18 Word images → \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\ada_output\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\word_media
[INFO] Alt text entries for 18 images
[INFO] Found 27 Excel images in \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\excel_images
[INFO] Total images to analyze: 45
[INFO] Unique image groups: 42
[WARN] Cannot read image: \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\ada_output\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\excel_images\Container_chart_1.png — cannot identify image file '\\\\wsl.localhost\\Ubuntu-24.04\\home\\joe\\work\\NotBic\\Ports\\ada_output\\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\\excel_images\\Container_chart_1.png'
[WARN] Cannot read image: \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\ada_output\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\excel_images\F 8_chart_4.png — cannot identify image file